In [1]:
###Lasso, ElasticNet & Sparsity###

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score

In [5]:
df = pd.read_csv("cp_data_demo.csv")

print(df.columns.tolist())

['Formula', 'Temperature', 'Heat_Capacity']


In [6]:
y = df["Heat_Capacity"]

X = df.drop(
    columns=["Formula", "Heat_Capacity"],
    errors="ignore"
)

X = X.select_dtypes(include=np.number)

# Missing feature values
X = X.fillna(X.median())

# Target me missing rows remove
valid = y.notna()

X = X.loc[valid]
y = y.loc[valid]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4570, 1)
y shape: (4570,)


In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score

# Load dataset
df = pd.read_csv("cp_data_demo.csv")

# Target
y = df["Heat_Capacity"]

# Features
X = df.drop(
    columns=["Formula", "Heat_Capacity"],
    errors="ignore"
)

# Only numeric features
X = X.select_dtypes(include=np.number)

# Fill missing feature values
X = X.fillna(X.median())

# Remove rows where target is missing
valid = y.notna()
X = X.loc[valid]
y = y.loc[valid]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Setup completed successfully!")

X_train: (3656, 1)
X_test: (914, 1)
Setup completed successfully!


In [10]:
# 1. Train Lasso & Extract Zeroed Features
lasso = Lasso(alpha=1.0, random_state=42)
lasso.fit(X_train, y_train)
lasso_preds = lasso.predict(X_test)

zeroed_features = X_train.columns[lasso.coef_ == 0]
print(f"Lasso forced {len(zeroed_features)} features to exactly 0.")
print("Sample of dropped features:", list(zeroed_features[:5]))

Lasso forced 0 features to exactly 0.
Sample of dropped features: []


In [11]:
# 2. Train ElasticNet
elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
elastic.fit(X_train, y_train)
elastic_preds = elastic.predict(X_test)

In [12]:
# 3. Define Evaluation Helper
def evaluate_model(y_true, y_pred):
    return {
        'R²': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred)
    }

In [14]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score

In [17]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

alphas = [0.01, 0.1, 1, 10, 100]

ridge_test_scores = []

for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    score = r2_score(
        y_test,
        model.predict(X_test)
    )

    ridge_test_scores.append(score)

best_index = np.argmax(ridge_test_scores)
best_alpha = alphas[best_index]

print("Best alpha:", best_alpha)
print("Best Test R²:", ridge_test_scores[best_index])

Best alpha: 0.01
Best Test R²: 0.014580051975909614


In [20]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

# --------------------------------
# 1. Linear Regression
# --------------------------------
lr = LinearRegression()
lr.fit(X_train, y_train)

# --------------------------------
# 2. Best Ridge Model
# --------------------------------
ridge_best = Ridge(alpha=best_alpha)
ridge_best.fit(X_train, y_train)

# --------------------------------
# 3. Lasso predictions
# --------------------------------
# lasso_preds should already exist
# If not, create Lasso model
if 'lasso_preds' not in globals():
    from sklearn.linear_model import Lasso
    
    lasso = Lasso(alpha=1.0, random_state=42)
    lasso.fit(X_train, y_train)
    lasso_preds = lasso.predict(X_test)

# --------------------------------
# 4. ElasticNet predictions
# --------------------------------
if 'elastic_preds' not in globals():
    from sklearn.linear_model import ElasticNet
    
    elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
    elastic.fit(X_train, y_train)
    elastic_preds = elastic.predict(X_test)

# --------------------------------
# 5. Evaluation Function
# --------------------------------
def evaluate_model(y_true, y_pred):
    return {
        'R² Score': r2_score(y_true, y_pred)
    }

# --------------------------------
# 6. Generate 4-Model Comparison
# --------------------------------
results = pd.DataFrame({
    'Unregularized LR': evaluate_model(
        y_test, lr.predict(X_test)
    ),

    'Ridge (L2)': evaluate_model(
        y_test, ridge_best.predict(X_test)
    ),

    'Lasso (L1)': evaluate_model(
        y_test, lasso_preds
    ),

    'ElasticNet': evaluate_model(
        y_test, elastic_preds
    )
}).T

print("\n--- 4-Model Evaluation Metrics ---")
display(results)


--- 4-Model Evaluation Metrics ---


,R² Score
Unregularized LR,0.014580
Ridge (L2),0.014580
Lasso (L1),0.014578
ElasticNet,0.014579
